# MELD 7-Class Emotion — MultiEMO Pre-Extracted Features (Exp 11)

**No extraction needed.** Downloads the ~213 MB feature pkls used by MultiEMO (ACL 2023) directly from their GitHub fork. No Google Drive, no HuggingFace rate limits, no GPU feature extraction.

**Features:**
- Text: **RoBERTa-Large 1024-dim** (frozen, pre-pooled per utterance)
- Audio: **openSMILE IS10 1582-dim** (utterance-level)
- Visual: **DenseNet 342-dim** (face-level, utterance-mean-pooled)

These are the **exact features** used by MultiEMO, MM-DFN, MMGCN, COGMEN (i.e., the MELD SOTA papers hitting ~65% F1w).

**Architecture (Exp 11 — Transformer encoders):** Each modality feature vector is split into `n_tokens` sub-vectors ("feature tokenisation"), a learnable positional embedding is added, then `n_enc_layers` Transformer encoder blocks (self-attention + FFN) refine the token sequence before mean-pooling to `d_m`. This replaces the 2-layer MLP used in Exp 10, allowing the encoder to model intra-feature interactions rather than just projecting.

**Setup on Kaggle:**
1. New Notebook → Settings → Accelerator → **GPU T4 x1**
2. Upload this notebook and Run All
3. No dataset attachment needed — features are downloaded at runtime

In [ ]:
!pip install -q tqdm scikit-learn

In [ ]:
import os, sys, math, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm
from types import SimpleNamespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')

## 1. Download MultiEMO Features (~213 MB, one-time)

In [ ]:
FEAT_DIR = '/kaggle/working/meld_feats'
os.makedirs(FEAT_DIR, exist_ok=True)

BASE = 'https://github.com/LuckyDaydreamer/MultiEMO/raw/main/Data/MELD'
FILES = ['AudioFeatures.pkl', 'VisualFeatures.pkl', 'TextFeatures.pkl', 'Speakers.pkl']

for fn in FILES:
    out = f'{FEAT_DIR}/{fn}'
    if os.path.exists(out):
        print(f'  {fn}: already downloaded ({os.path.getsize(out)/1e6:.1f} MB)')
        continue
    print(f'Downloading {fn}...')
    !wget -q -O {out} {BASE}/{fn}
    print(f'  -> {os.path.getsize(out)/1e6:.1f} MB')

!ls -lh {FEAT_DIR}

## 2. Inspect Feature Structure
The pkls are nested dicts keyed by dialogue ID. Let's see exactly what's in them.

In [ ]:
def load_pkl(name):
    with open(f'{FEAT_DIR}/{name}', 'rb') as f:
        return pickle.load(f)

textFeatures   = load_pkl('TextFeatures.pkl')
audioFeatures  = load_pkl('AudioFeatures.pkl')
visualFeatures = load_pkl('VisualFeatures.pkl')
speakersData   = load_pkl('Speakers.pkl')

def describe(name, obj, depth=0, max_depth=3):
    indent = '  ' * depth
    if isinstance(obj, dict):
        print(f'{indent}{name}: dict, {len(obj)} keys')
        if depth < max_depth:
            sample_key = next(iter(obj.keys()))
            print(f'{indent}  sample key: {repr(sample_key)[:60]}')
            describe('[val]', obj[sample_key], depth + 1, max_depth)
    elif isinstance(obj, (list, tuple)):
        print(f'{indent}{name}: {type(obj).__name__}, len={len(obj)}')
        if len(obj) > 0 and depth < max_depth:
            describe('[0]', obj[0], depth + 1, max_depth)
    elif hasattr(obj, 'shape'):
        print(f'{indent}{name}: {type(obj).__name__} shape={obj.shape} dtype={getattr(obj, "dtype", "?")}')
    else:
        s = repr(obj)[:80]
        print(f'{indent}{name}: {type(obj).__name__} = {s}')

print('=== TextFeatures ===')
describe('TextFeatures', textFeatures)
print('\n=== AudioFeatures ===')
describe('AudioFeatures', audioFeatures)
print('\n=== VisualFeatures ===')
describe('VisualFeatures', visualFeatures)
print('\n=== Speakers.pkl ===')
describe('Speakers', speakersData)

## 3. Parse Speakers.pkl & Build Per-Utterance Dataset

The MultiEMO/MM-DFN convention stores `Speakers.pkl` as either:
- A tuple `(videoSpeakers, videoLabels, trainVid, testVid, validVid)`
- Or a dict with keys like `'videoSpeakers', 'videoLabels', 'trainVid', 'testVid', 'validVid'`

We handle both. Each split is a set of dialogue IDs; each dialogue has a list of per-utterance features and labels.

In [ ]:
# Parse Speakers.pkl defensively — it may be a tuple or dict
def parse_speakers(obj):
    if isinstance(obj, dict):
        return (obj.get('videoSpeakers') or obj.get('speakers'),
                obj.get('videoLabels') or obj.get('labels'),
                obj.get('trainVid')    or obj.get('train'),
                obj.get('validVid')    or obj.get('valid') or obj.get('devVid') or obj.get('dev'),
                obj.get('testVid')     or obj.get('test'))
    if isinstance(obj, (list, tuple)):
        # Common orderings across MELD repos
        if len(obj) == 5:
            # (speakers, labels, train, test, valid)  — MultiEMO style
            return obj[0], obj[1], obj[2], obj[4], obj[3]
        if len(obj) == 4:
            # (speakers, labels, train, test) — valid missing, borrow from train
            return obj[0], obj[1], obj[2], None, obj[3]
    raise ValueError(f'Unrecognised Speakers.pkl format: {type(obj)} len={len(obj) if hasattr(obj, "__len__") else "?"}')

videoSpeakers, videoLabels, trainVid, validVid, testVid = parse_speakers(speakersData)

def _sz(x):
    if x is None: return 'None'
    if hasattr(x, '__len__'): return len(x)
    return '?'

print(f'videoSpeakers : {type(videoSpeakers).__name__}, {_sz(videoSpeakers)} dialogues')
print(f'videoLabels   : {type(videoLabels).__name__}, {_sz(videoLabels)} dialogues')
print(f'trainVid      : {_sz(trainVid)}')
print(f'validVid      : {_sz(validVid)}')
print(f'testVid       : {_sz(testVid)}')

# If validVid missing, split trainVid 90/10
if validVid is None or len(validVid) == 0:
    print('\nvalidVid missing — splitting trainVid 90/10')
    rng = random.Random(42)
    train_list = list(trainVid)
    rng.shuffle(train_list)
    n_val = max(1, len(train_list) // 10)
    validVid = set(train_list[:n_val])
    trainVid = set(train_list[n_val:])
    print(f'  new trainVid: {len(trainVid)}, new validVid: {len(validVid)}')

In [ ]:
# Flatten dialogue-level features to per-utterance samples
EMOTION_LABELS = ['Neutral', 'Surprise', 'Fear', 'Sadness', 'Joy', 'Disgust', 'Anger']
NUM_CLASSES = 7

def to_numpy(x):
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    if hasattr(x, 'numpy'):
        return x.detach().cpu().numpy().astype(np.float32)
    return np.asarray(x, dtype=np.float32)

def flatten_split(dialogue_ids):
    out = []
    missing = 0
    for dia_id in dialogue_ids:
        if dia_id not in videoLabels or dia_id not in textFeatures:
            missing += 1
            continue
        labels_d = videoLabels[dia_id]
        text_d   = textFeatures[dia_id]
        audio_d  = audioFeatures[dia_id]
        video_d  = visualFeatures[dia_id]
        n = len(labels_d)
        for i in range(n):
            out.append({
                'text':  to_numpy(text_d[i]),
                'audio': to_numpy(audio_d[i]),
                'video': to_numpy(video_d[i]),
                'label': int(labels_d[i]),
            })
    if missing:
        print(f'  (missing dialogues skipped: {missing})')
    return out

train_data = flatten_split(trainVid)
dev_data   = flatten_split(validVid)
test_data  = flatten_split(testVid)

print(f'Train={len(train_data)}, Dev={len(dev_data)}, Test={len(test_data)}')

# Sanity check dimensions
s = train_data[0]
print(f'\nSample 0 dims:')
print(f'  text : {s["text"].shape}')
print(f'  audio: {s["audio"].shape}')
print(f'  video: {s["video"].shape}')
print(f'  label: {s["label"]} ({EMOTION_LABELS[s["label"]]})')

In [ ]:
# Label distribution per split
def count_labels(data):
    c = np.zeros(NUM_CLASSES, dtype=int)
    for s in data: c[s['label']] += 1
    return c

print('Label distribution:')
print(f'  {"":<10} ' + ' '.join(f'{e[:5]:>7}' for e in EMOTION_LABELS))
for name, d in [('train', train_data), ('dev', dev_data), ('test', test_data)]:
    c = count_labels(d)
    print(f'  {name:<10} ' + ' '.join(f'{x:>7d}' for x in c))

# Infer feature dimensions from data
TEXT_DIM  = train_data[0]['text'].shape[-1]
AUDIO_DIM = train_data[0]['audio'].shape[-1]
VIDEO_DIM = train_data[0]['video'].shape[-1]
print(f'\nFeature dims — text:{TEXT_DIM}, audio:{AUDIO_DIM}, video:{VIDEO_DIM}')

## 4. Config

In [ ]:
config = SimpleNamespace(
    modalities=['text', 'audio', 'video'],

    text_dim  = TEXT_DIM,
    audio_dim = AUDIO_DIM,
    video_dim = VIDEO_DIM,

    d_m        = 128,
    d_ff       = 256,
    num_classes= NUM_CLASSES,

    # Transformer utterance encoder
    n_tokens        = 4,    # feature tokenisation: split each modality vector into n_tokens chunks
    n_enc_layers    = 2,    # number of transformer encoder layers inside each modality encoder

    cross_att_heads = 4,
    att_dropout     = 0.2,
    dropout         = 0.1,

    batch_size       = 64,
    lr               = 1e-3,
    epochs           = 50,
    early_stop       = 10,
    grad_clip        = 1.0,
    label_smoothing  = 0.1,
    use_class_weights= True,
    use_lr_scheduler = True,
)
print(config)

## 5. Model — Transformer Encoders + MTFN Cross-Attention

Each modality is already an utterance-level vector. Instead of projecting directly with an MLP, we "tokenise" the feature vector by splitting it into `n_tokens` equal chunks (via a linear map to `n_tokens × d_m`), add learnable positional embeddings, then run `n_enc_layers` Transformer encoder blocks (self-attention + FFN). The mean-pooled output is a `d_m`-dim utterance embedding, which is fed as a length-1 token into MTFN. MTFN computes 6 cross-modal attention pairs → fusion → encoder-decoder → predictions.

In [ ]:
class TransformerUtteranceEncoder(nn.Module):
    """
    Feature-tokenisation encoder: (B, input_dim) → (B, 1, d_m) + (B, num_classes).

    1. Linear projects input_dim → n_tokens * d_m, reshaped to (B, n_tokens, d_m).
    2. Adds learnable positional embeddings.
    3. Applies n_enc_layers Transformer encoder blocks (self-attention + FFN).
    4. Mean-pools tokens → (B, d_m) for the unimodal prediction head.
    5. Returns unsqueezed pooled vector as a length-1 sequence for MTFN.
    """
    def __init__(self, input_dim, d_m, num_classes, n_tokens=4, n_enc_layers=2,
                 n_heads=4, d_ff=256, dropout=0.1):
        super().__init__()
        self.n_tokens = n_tokens
        self.d_m = d_m
        self.tokenizer = nn.Linear(input_dim, n_tokens * d_m)
        self.pos_emb = nn.Parameter(torch.zeros(1, n_tokens, d_m))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_m, n_heads, d_ff, dropout) for _ in range(n_enc_layers)
        ])
        self.norm = nn.LayerNorm(d_m)
        self.pred_head = nn.Linear(d_m, num_classes)

    def forward(self, x):
        tokens = self.tokenizer(x).reshape(x.size(0), self.n_tokens, self.d_m)  # (B, n_tokens, d_m)
        tokens = tokens + self.pos_emb
        for layer in self.encoder_layers:
            tokens = layer(tokens)
        h = self.norm(tokens.mean(dim=1))          # (B, d_m)
        return h.unsqueeze(1), self.pred_head(h)   # (B, 1, d_m), (B, C)


class CrossModalAttention(nn.Module):
    def __init__(self, d_m, n_heads, att_dropout=0.2):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_m, n_heads, dropout=att_dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_m)

    def forward(self, x_i, x_j):
        out, _ = self.mha(x_i, x_j, x_j)
        return self.layer_norm(out)


class EncoderBlock(nn.Module):
    def __init__(self, d_m, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_m, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_m, d_ff), nn.ReLU(), nn.Linear(d_ff, d_m))
        self.norm1 = nn.LayerNorm(d_m)
        self.norm2 = nn.LayerNorm(d_m)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.drop(a))
        return self.norm2(x + self.drop(self.ff(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_m, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_m, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_m, d_ff), nn.ReLU(), nn.Linear(d_ff, d_m))
        self.norm1 = nn.LayerNorm(d_m)
        self.norm2 = nn.LayerNorm(d_m)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory):
        a, _ = self.cross_attn(x, memory, memory)
        x = self.norm1(x + self.drop(a))
        return self.norm2(x + self.drop(self.ff(x)))


class MTFN(nn.Module):
    """Modality-agnostic MTFN. Handles any subset of modalities."""
    def __init__(self, d_m, num_classes, modalities, n_heads=4, d_ff=256, dropout=0.1, att_dropout=0.2):
        super().__init__()
        self.modalities = modalities
        self.projs = nn.ModuleDict({m: nn.Linear(d_m, d_m) for m in modalities})
        self.cross_atts = nn.ModuleDict({
            f'{mi}2{mj}': CrossModalAttention(d_m, n_heads, att_dropout)
            for mi in modalities for mj in modalities if mi != mj
        })
        n_pairs = len(modalities) * (len(modalities) - 1)
        self.encoder = EncoderBlock(d_m, n_heads, d_ff, dropout)
        self.decoder = DecoderBlock(d_m, n_heads, d_ff, dropout)
        self.pred_fusion = nn.Linear(n_pairs * d_m, num_classes)
        self.pred_recon  = nn.Linear(d_m, num_classes)

    def forward(self, feats):
        proj = {m: self.projs[m](feats[m]) for m in self.modalities}
        tokens = []
        for mi in self.modalities:
            for mj in self.modalities:
                if mi != mj:
                    ca = self.cross_atts[f'{mi}2{mj}'](proj[mi], proj[mj])
                    tokens.append(ca.squeeze(1))   # length-1 seq → (B, d_m)
        tokens = torch.stack(tokens, dim=1)        # (B, n_pairs, d_m)
        y_m = self.pred_fusion(tokens.reshape(tokens.size(0), -1))
        enc_out = self.encoder(tokens)
        dec_out = self.decoder(tokens, enc_out)
        y_m_prime = self.pred_recon(dec_out.mean(dim=1))
        return y_m, y_m_prime


class MultiModalModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.modalities = cfg.modalities
        C = cfg.num_classes
        dims = {'text': cfg.text_dim, 'audio': cfg.audio_dim, 'video': cfg.video_dim}
        self.encs = nn.ModuleDict({
            m: TransformerUtteranceEncoder(
                dims[m], cfg.d_m, C,
                n_tokens=cfg.n_tokens,
                n_enc_layers=cfg.n_enc_layers,
                n_heads=cfg.cross_att_heads,
                d_ff=cfg.d_ff,
                dropout=cfg.dropout,
            ) for m in self.modalities
        })
        self.mtfn = MTFN(cfg.d_m, C, self.modalities,
                         n_heads=cfg.cross_att_heads, d_ff=cfg.d_ff,
                         dropout=cfg.dropout, att_dropout=cfg.att_dropout) \
                    if len(self.modalities) >= 2 else None

    def forward(self, text, audio, video):
        inputs = {'text': text, 'audio': audio, 'video': video}
        feats, preds = {}, {}
        for m in self.modalities:
            seq, y = self.encs[m](inputs[m])
            feats[m] = seq
            preds[m] = y
        if self.mtfn is not None:
            y_m, y_m_prime = self.mtfn(feats)
            preds['fusion'], preds['recon'] = y_m, y_m_prime
        else:
            sole = preds[self.modalities[0]]
            preds['fusion'] = preds['recon'] = sole
        return preds

## 6. Dataset & Training Loop

In [ ]:
class MELDFeatureDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        s = self.data[i]
        return (
            torch.from_numpy(s['text']),
            torch.from_numpy(s['audio']),
            torch.from_numpy(s['video']),
            torch.tensor(s['label'], dtype=torch.long),
        )

def make_loader(data, shuffle):
    return DataLoader(MELDFeatureDataset(data), batch_size=config.batch_size,
                      shuffle=shuffle, num_workers=2, pin_memory=True)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def compute_class_weights(data):
    counts = np.zeros(NUM_CLASSES)
    for s in data: counts[s['label']] += 1
    w = 1.0 / np.clip(counts, 1, None)
    return torch.FloatTensor(w / w.mean())

def compute_metrics(p, l):
    return {
        'Accuracy':    accuracy_score(l, p) * 100,
        'F1_weighted': f1_score(l, p, average='weighted', zero_division=0) * 100,
        'F1_macro':    f1_score(l, p, average='macro',    zero_division=0) * 100,
    }

def evaluate(model, loader):
    model.eval()
    use_amp = torch.cuda.is_available()
    all_p, all_l = [], []
    with torch.no_grad():
        for text, audio, video, labels in loader:
            text, audio, video = text.to(device), audio.to(device), video.to(device)
            with autocast('cuda', enabled=use_amp):
                preds = model(text, audio, video)
            all_p.append(preds['recon'].argmax(1).cpu().numpy())
            all_l.append(labels.numpy())
    p, l = np.concatenate(all_p), np.concatenate(all_l)
    return compute_metrics(p, l), p, l

def print_full_report(p, l):
    print('\n--- Classification Report ---')
    print(classification_report(l, p, target_names=EMOTION_LABELS, zero_division=0))
    print('--- Confusion Matrix ---')
    cm = confusion_matrix(l, p)
    print('          ' + ' '.join(f'{EMOTION_LABELS[i][:4]:>5}' for i in range(NUM_CLASSES)))
    for i in range(NUM_CLASSES):
        print(f'{EMOTION_LABELS[i]:<10}' + ' '.join(f'{cm[i,j]:5d}' for j in range(NUM_CLASSES)))

In [ ]:
set_seed(42)

train_loader = make_loader(train_data, shuffle=True)
dev_loader   = make_loader(dev_data,   shuffle=False)
test_loader  = make_loader(test_data,  shuffle=False)

class_weights = compute_class_weights(train_data).to(device) if config.use_class_weights else None
if class_weights is not None:
    print(f'Class weights: {class_weights.cpu().numpy().round(3)}')

model = MultiModalModel(config).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

optimizer = Adam(model.parameters(), lr=config.lr)
scheduler = LambdaLR(
    optimizer,
    lambda t: max(0., 0.5 * (1 + math.cos(math.pi * t / max(1, config.epochs))))
) if config.use_lr_scheduler else None

ce_loss = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config.label_smoothing)
print('Loss: CrossEntropyLoss')

In [ ]:
best_dev_f1, best_epoch = 0.0, 0
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
CKPT = '/kaggle/working/checkpoints/best_multiemo.pt'

use_amp = torch.cuda.is_available()
scaler  = GradScaler('cuda', enabled=use_amp)

for epoch in range(1, config.epochs + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}', leave=False, mininterval=1.0)
    for text, audio, video, labels in pbar:
        text, audio, video, labels = text.to(device), audio.to(device), video.to(device), labels.to(device)
        with autocast('cuda', enabled=use_amp):
            preds = model(text, audio, video)
            loss = sum(ce_loss(preds[k], labels) for k in list(config.modalities) + ['fusion', 'recon'])
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    if scheduler is not None:
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    dev_metrics, _, _ = evaluate(model, dev_loader)
    print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | Acc={dev_metrics['Accuracy']:.1f} | "
          f"F1w={dev_metrics['F1_weighted']:.1f} | F1m={dev_metrics['F1_macro']:.1f}")

    if dev_metrics['F1_weighted'] > best_dev_f1:
        best_dev_f1, best_epoch = dev_metrics['F1_weighted'], epoch
        torch.save(model.state_dict(), CKPT)
        print(f'  -> New best Dev F1w={best_dev_f1:.2f} — saved.')
    elif (epoch - best_epoch) >= config.early_stop:
        print(f'Early stopping at epoch {epoch} (best: {best_epoch})')
        break

print(f'\nLoading best checkpoint (epoch {best_epoch})...')
model.load_state_dict(torch.load(CKPT, map_location=device))
test_metrics, test_preds, test_labels = evaluate(model, test_loader)

print('\n========== Test Results ==========')
print(f"Accuracy       : {test_metrics['Accuracy']:.1f}")
print(f"F1 (weighted)  : {test_metrics['F1_weighted']:.1f}")
print(f"F1 (macro)     : {test_metrics['F1_macro']:.1f}")
print('==================================')
print_full_report(test_preds, test_labels)